In [ ]:
############## Tokenizers Initialization ##############
from Tokenizers.tokenizer import BasePoetryTokenizer, WordPieceTokenizer, SyllableTokenizer
from Configuration import PATH_CONFIGURATION, TOKENIZER_CONFIGURATION
import os

############## CONFIGURATION ##############
SYLLABLE_TOKENIZER_PATH = PATH_CONFIGURATION.TOKENIZERS_PATH()["SYLLABLE"]
WORDPIECE_TOKENIZER_PATH = PATH_CONFIGURATION.TOKENIZERS_PATH()["WORDPIECE"]
MAX_WORD_DICTIONARY_SIZE = TOKENIZER_CONFIGURATION.PARAMETERS()["WORDPIECE_VOCAB_SIZE"]


In [5]:
def process_and_save(
    json_dir: str,
    tokenizer: BasePoetryTokenizer,
    output_filename: str,
    max_word_dictionary: int,
    max_words: int = None
):
    """
        Processes a directory of JSON files containing poems, trains the tokenizer on the collected poems, 
        and saves the trained tokenizer to the specified output file.

        Args:
             json_dir (str):The directory containing JSON files with poems.
             tokenizer (BasePoetryTokenizer): An instance of a tokenizer to be trained.
             output_filename (str): The filename where the trained tokenizer will be saved.
             max_word_dictionary (int): The maximum size of the tokenizer's vocabulary.
    """
    if not os.path.exists(json_dir):
        raise FileNotFoundError(f"La directory '{json_dir}' non esiste.")
        
    tokenizer.dict_path = output_filename #type: ignore

    json_files = [f for f in sorted(os.listdir(json_dir)) if f.endswith(".json")]
    print(f"[INFO] Trovati {len(json_files)} file JSON. Lettura corpus in corso...")

    tutte_le_poesie = []
    for filename in json_files:
        file_path = os.path.join(json_dir, filename)
        tutte_le_poesie.extend(tokenizer.parse_json_poems(file_path))


    tokenizer.train_from_corpus(tutte_le_poesie, vocab_size=max_word_dictionary)
    if max_words is not None and tokenizer is SyllableTokenizer:
        tokenizer.truncate_vocab(max_words)
    tokenizer.save_tokenizer()
    print(f"[SUCCESS] Dictionary saved successfully in '{output_filename}'.")

In [6]:
############################ SYLLABLE TOKENIZER INITIALIZATION ########################
import os
if not os.path.exists(SYLLABLE_TOKENIZER_PATH):
    syllable_tokenizer = SyllableTokenizer(dict_path=SYLLABLE_TOKENIZER_PATH)
    process_and_save(
        json_dir="../biblioteca_italiana/json",
        tokenizer=syllable_tokenizer,
        output_filename=SYLLABLE_TOKENIZER_PATH,
        max_word_dictionary=16561
    )


# If the dictionary already exists, we can load it directly without retraining.
if os.path.exists(SYLLABLE_TOKENIZER_PATH):
    syllable_tokenizer = SyllableTokenizer(dict_path=SYLLABLE_TOKENIZER_PATH)

print("\n[SUCCESS] Syllable tokenizer initialized successfully.")



[SUCCESS] Syllable tokenizer initialized successfully.


In [ ]:
############################# WORDPIECE TOKENIZER INITIALIZATION ########################
if not os.path.exists(WORDPIECE_TOKENIZER_PATH):
    wordpiece_tokenizer = WordPieceTokenizer(dict_path=WORDPIECE_TOKENIZER_PATH)
    process_and_save(
        json_dir="../biblioteca_italiana/json",
        tokenizer=wordpiece_tokenizer,
        output_filename=WORDPIECE_TOKENIZER_PATH,
        max_word_dictionary=MAX_WORD_DICTIONARY_SIZE
    )

# If the dictionary already exists, we can load it directly without retraining.
if os.path.exists(WORDPIECE_TOKENIZER_PATH):
    wordpiece_tokenizer = WordPieceTokenizer(dict_path=WORDPIECE_TOKENIZER_PATH)

print("\n[SUCCESS] WordPiece tokenizer initialized successfully.")


[SUCCESS] Inizializzazione del WordPieceTokenizer completata con successo.


In [4]:
# Tokenization example

import json

path = '../biblioteca_italiana/json/dante_alighieri.json'
adapted_poem = BasePoetryTokenizer.parse_json_poems(path)
print(f"Adapted Poem:\n{adapted_poem}\n")
print(f"Syllabify poem:\n{syllable_tokenizer.syllabify_poem(adapted_poem[0])}\n")
print(f"Tokenized Poem:\n{syllable_tokenizer.tokenize(adapted_poem[0])}\n")
print(f"Detokenized Poem:\n{syllable_tokenizer.detokenize(syllable_tokenizer.tokenize(adapted_poem[0]), as_text=True, control_tokens=True)}\n")
print(f"Detokenized Poem:\n{syllable_tokenizer.detokenize(syllable_tokenizer.tokenize(adapted_poem[0]), as_text=False, control_tokens=False)}\n")

Adapted Poem:
['Nel mezzo del cammin di nostra vita [VERSE] mi ritrovai per una selva oscura, [VERSE] ché la diritta via era smarrita. [STANZA] Ahi quanto a dir qual era è cosa dura [VERSE] esta selva selvaggia e aspra e forte [VERSE] che nel pensier rinova la paura! [STANZA] Tant\'è amara che poco è più morte; [VERSE] ma per trattar del ben ch\'i\' vi trovai, [VERSE] dirò de l\'altre cose ch\'i\' v\'ho scorte. [STANZA] Io non so ben ridir com\'i\' v\'intrai, [VERSE] tant\'era pien di sonno a quel punto [VERSE] che la verace via abbandonai. [STANZA] Ma poi ch\'i\' fui al piè d\'un colle giunto, [VERSE] là dove terminava quella valle [VERSE] che m\'avea di paura il cor compunto, [STANZA] guardai in alto, e vidi le sue spalle [VERSE] vestite già de\' raggi del pianeta [VERSE] che mena dritto altrui per ogne calle. [STANZA] Allor fu la paura un poco queta, [VERSE] che nel lago del cor m\'era durata [VERSE] la notte ch\'i\' passai con tanta pieta. [STANZA] E come quei che con lena affannat